# SpecDist — Kaggle Quickstart

**Free GPU · 30 h/week · T4 (16 GB VRAM) · 9-hour sessions · 29 GB system RAM**

| Cell | What it does | Time |
|------|-------------|------|
| 0. Bootstrap | One-shot: clone → deps → auth → run | ~3 min to start |
| 1. Resume | After session restart — pipeline skips completed steps | ~1 min + rest |
| 2. Monitor | State + log tail (auto-refresh option) | instant |
| 3. Save | Verify artifacts before session ends | instant |

---

### Before you start
1. **GPU**: Settings (right sidebar) → Accelerator → **GPU T4 × 1** (or T4 × 2 — see below)
2. **Internet**: Settings → Internet → **On**
3. **Kaggle Secrets** (Add-ons → Secrets):
   - `WANDB_API_KEY` — https://wandb.ai/authorize
   - `HF_TOKEN` — https://huggingface.co/settings/tokens _(Qwen3 models are public — optional)_
   - `GITHUB_TOKEN` — only if the repo is private
4. **Run all**: Shift+F5

### CONFIG options

| CONFIG | Teacher | Steps | VRAM | Time on T4 | Notes |
|--------|---------|-------|------|-----------|-------|
| `colab_lite` | Qwen3-1.7B (BF16) | 300 | ~5.6 GB | ~25 min | Quick trend check |
| `colab` | Qwen3-4B (BF16) | 500 | ~10.7 GB | ~4 h | Safe — no quantization |
| `kaggle` | Qwen3-8B (4-bit NF4) | 1000 | ~7.8 GB | ~5-8 h | **Best for Kaggle** — 8B signal, fits 9h session |

**Why `kaggle` config works here but not on Colab:**  
4-bit NF4 loading needs ~16 GB CPU RAM for Qwen3-8B tensor intermediates.  
Colab has ~12 GB RAM → OOM kill. Kaggle has **29 GB RAM** → succeeds.  
Same 8B teacher as the A100 paper runs, but quantized to fit T4 VRAM.

### Session timeouts
- **60 min idle** (browser open): prevented by the keep-alive JS in Cells 0 and 1
- **60 min idle** (browser closed): cannot be prevented — keep the tab open
- **9-hour absolute limit**: cannot be extended — set `BACKGROUND=True`, then run Cell 1 in the next session

### Dual GPU (T4 × 2)
Kaggle lets you enable **two T4s for free** (Settings → Accelerator → GPU T4 × 2).  
With `CONFIG = "kaggle"` (4-bit NF4), `device_map="auto"` automatically shards the  
8B teacher across both GPUs — no code changes needed. Even more VRAM headroom per GPU.

### Storage
- `/kaggle/working/` — **20 GB per session** (wiped on restart)
- Persist checkpoints: after each session → **Notebook → Data → Output → + New Dataset** → name it `specdist-checkpoints`
- Next session: **Add Data → Your Datasets** → attach it → Cell 1 restores it automatically

### Pre-uploading model weights (zero download time)
1. Download locally: `huggingface-cli download Qwen/Qwen3-0.6B Qwen/Qwen3-8B --ignore-patterns '*.gguf' '*.bin'`
2. Upload `~/.cache/huggingface/` as a private Kaggle dataset (100 GB total quota, 20 GB per dataset — 8B model ~15 GB fits in one)
3. Attach to this notebook → set `KAGGLE_HF_DATASET = "/kaggle/input/<your-dataset-name>"` in Cell 0
4. Models are available **instantly** at session start — no download ever again

Full setup guide: `docs/KAGGLE.md`

In [ ]:
# =============================================================================
# Cell 0 — BOOTSTRAP  (run once per fresh session)
# Edit the variables below, then Shift+F5 (Run All).
# After a session restart use Cell 1 (Resume) instead — it is self-contained.
# =============================================================================

# -- Edit these ---------------------------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR     = "/kaggle/working/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"

# CONFIG options:
#   kaggle     — Qwen3-8B (4-bit NF4, ~7.8 GB VRAM, 1000 steps, ~5-8 h)  ← best for Kaggle
#   colab      — Qwen3-4B (BF16,      ~10.7 GB VRAM, 500 steps,  ~4 h)    ← safe / no quant
#   colab_lite — Qwen3-1.7B (BF16,    ~5.6 GB VRAM,  300 steps,  ~25 min) ← quick trend check
CONFIG      = "kaggle"
SMOKE       = False    # True = 10-step crash check (~5 min); run this first on a new setup
BACKGROUND  = True     # True = pipeline runs in background; monitor with Cell 2
LOSSES      = None     # None = all losses  |  "kl,ebe" = subset
EXTRA_ARGS  = []

# Optional: path to a pre-attached Kaggle dataset containing HF model weights.
# Upload ~/.cache/huggingface/ as a private Kaggle dataset (once), attach it
# to this notebook, then set this to skip the ~5-10 min model download every session.
# See docs/KAGGLE.md for detailed instructions.
# Example: KAGGLE_HF_DATASET = "/kaggle/input/qwen3-hf-cache"
KAGGLE_HF_DATASET = None
# -----------------------------------------------------------------------------

import os, subprocess, sys

# -- Keep-alive: prevent 60-min idle timeout (45 s JS heartbeat) --------------
# Fires a synthetic mousemove event so Kaggle thinks the session is active.
# Keeps the session alive up to the 9-hour absolute limit.
# Only helps if the browser tab stays open — close the tab and idle timeout applies.
try:
    from IPython.display import display, Javascript
    display(Javascript("""
(function(){
  if(window.__sd_ka)return;
  window.__sd_ka=setInterval(function(){
    document.dispatchEvent(new MouseEvent('mousemove',{bubbles:true}));
  },45000);
  console.log('[specdist] keep-alive on (45 s heartbeat)');
})();
"""))
    print("[keep-alive] 45 s JS heartbeat started (prevents 60-min idle timeout)")
except Exception:
    pass  # not in a browser context — silently skip

# -- Kaggle secrets → env  (must run before any colab_utils import) -----------
def _kread(name):
    """Read a Kaggle secret; fall back to environment variable."""
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

for _k in ("WANDB_API_KEY", "HF_TOKEN", "GITHUB_TOKEN"):
    _v = _kread(_k)
    if _v:
        os.environ[_k] = _v
        print(f"  {_k}: set")
    else:
        print(f"  {_k}: not found (Add-ons -> Secrets)")

# -- Clone / update repo -------------------------------------------------------
gh = os.environ.get("GITHUB_TOKEN", "")
clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh:  # keep token current in remote URL in case it rotated
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url],
                       capture_output=True)
    pull_r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                             capture_output=True, text=True)
    print(pull_r.stdout.strip() or "Repo already up to date")
    if pull_r.returncode != 0:
        print("[pull error]", pull_r.stderr.strip())

# Flush stale module cache so updated colab_utils is loaded fresh after pull.
# Without this, Python serves the old in-memory module even if the file changed.
sys.modules.pop("colab_utils", None)

# -- Dirs ----------------------------------------------------------------------
os.makedirs(f"{STORAGE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{STORAGE_ROOT}/logs", exist_ok=True)
os.chdir(GBV_DIR)
sys.path.insert(0, f"{GBV_DIR}/deploy")

# -- Import shared utils (repo is current, module cache flushed) ---------------
from colab_utils import (install_deps, fetch_training_data, setup_hf_cache,
                          prefetch_models, auth_wandb, auth_hf, check_gpu,
                          run_pipeline)

# -- Install deps (pip packages are wiped on every session restart) ------------
install_deps(GBV_DIR)

# -- HF model cache ------------------------------------------------------------
if KAGGLE_HF_DATASET and os.path.isdir(KAGGLE_HF_DATASET):
    # Pre-attached dataset: models available instantly, zero download.
    # Dataset must be structured as a standard HF Hub cache directory.
    # See docs/KAGGLE.md for how to create and upload this dataset.
    os.environ["HF_HOME"]            = KAGGLE_HF_DATASET
    os.environ["TRANSFORMERS_CACHE"] = KAGGLE_HF_DATASET
    for _flag in ("TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_OFFLINE"):
        os.environ.pop(_flag, None)
    print(f"[cache] Using attached dataset: {KAGGLE_HF_DATASET}")
else:
    # Download to /kaggle/working/specdist/hf_cache/ (ephemeral, ~20 GB limit).
    # kaggle config: 8B NF4 (~15 GB) + 0.6B (~1.2 GB) — takes ~5-10 min first session.
    setup_hf_cache(STORAGE_ROOT)
    prefetch_models(CONFIG, GBV_DIR, STORAGE_ROOT)

# -- Auth + training data ------------------------------------------------------
auth_wandb()
auth_hf()
fetch_training_data(GBV_DIR)   # downloads gsm8k_train.jsonl if missing (~3 MB)
check_gpu(warn_below_gb=12.0)  # warns if GPU not available
print()

# -- Run pipeline --------------------------------------------------------------
# background=True: cell returns immediately; use Cell 2 (Monitor) to track progress.
# background=False: cell blocks until the full pipeline completes (ok for smoke tests).
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR,
                 smoke=SMOKE, losses=LOSSES,
                 background=BACKGROUND, extra_args=EXTRA_ARGS)

In [ ]:
# =============================================================================
# Cell 1 — RESUME  (after session restart, idle timeout, or 9-hour limit)
# Self-contained: works correctly even if Cell 0 never ran this session.
# The pipeline reads the state file and skips already-completed steps.
# =============================================================================

# -- Edit these (must match Cell 0) -------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR     = "/kaggle/working/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"
CONFIG       = "kaggle"   # must match the CONFIG used in Cell 0
KAGGLE_HF_DATASET = None  # same as Cell 0
# -----------------------------------------------------------------------------

import os, subprocess, sys, shutil

# -- Keep-alive (45 s heartbeat — prevents 60-min idle timeout) ---------------
try:
    from IPython.display import display, Javascript
    display(Javascript("""
(function(){
  if(window.__sd_ka)return;
  window.__sd_ka=setInterval(function(){
    document.dispatchEvent(new MouseEvent('mousemove',{bubbles:true}));
  },45000);
  console.log('[specdist] keep-alive on (45 s heartbeat)');
})();
"""))
    print("[keep-alive] 45 s JS heartbeat started")
except Exception:
    pass

# -- Secrets → env -------------------------------------------------------------
def _kread(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

for _k in ("WANDB_API_KEY", "HF_TOKEN", "GITHUB_TOKEN"):
    _v = _kread(_k)
    if _v:
        os.environ[_k] = _v

# -- Clone / update repo -------------------------------------------------------
gh = os.environ.get("GITHUB_TOKEN", "")
clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url],
                       capture_output=True)
    pull_r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                             capture_output=True, text=True)
    print(pull_r.stdout.strip() or "Repo already up to date")
    if pull_r.returncode != 0:
        print("[pull error]", pull_r.stderr.strip())

# Flush stale module cache — without this, Python serves the old in-memory
# module even if the file on disk changed after the pull.
sys.modules.pop("colab_utils", None)

# -- Reinstall deps (always wiped on session restart) --------------------------
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", f"{REPO_DIR}/gbv-research/requirements.txt",
     "bitsandbytes", "accelerate", "torchao>=0.16.0"],
    check=True,
)
print("Deps installed")

# -- Restore checkpoints from an attached input dataset (if any) ---------------
# If you saved checkpoints as a Kaggle dataset (Cell 3 instructions) and
# re-attached it via Add Data -> Your Datasets, this restores them.
# Tries common dataset name variants automatically.
os.makedirs(f"{STORAGE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{STORAGE_ROOT}/logs", exist_ok=True)

_restored = False
for _ds_name in ("specdist-checkpoints", "specdist_checkpoints",
                 "specdist-output",      "specdist"):
    _ds_path = f"/kaggle/input/{_ds_name}"
    if os.path.isdir(_ds_path):
        _ckpt_dst = f"{STORAGE_ROOT}/checkpoints"
        # Restore results.db if not already present
        _db_src = os.path.join(_ds_path, "specdist", "results.db")
        _db_dst = os.path.join(STORAGE_ROOT, "results.db")
        if os.path.exists(_db_src) and not os.path.exists(_db_dst):
            shutil.copy2(_db_src, _db_dst)
            print(f"Restored results.db from {_db_src}")
        # Restore checkpoint directories
        for _item in os.listdir(_ds_path):
            _src = os.path.join(_ds_path, _item)
            _dst = os.path.join(_ckpt_dst, _item)
            if not os.path.exists(_dst):
                if os.path.isdir(_src):
                    shutil.copytree(_src, _dst)
                else:
                    shutil.copy2(_src, _dst)
        _items = os.listdir(_ckpt_dst)
        print(f"Checkpoints restored from {_ds_path}: {_items}")
        _restored = True
        break

if not _restored:
    print("No checkpoint dataset attached — starting from scratch")
    print("  (To attach: Notebook -> Data -> + Add Data -> Your Datasets)")

# -- Paths + sys.path ----------------------------------------------------------
os.chdir(GBV_DIR)
sys.path.insert(0, f"{GBV_DIR}/deploy")

# -- Import shared utils -------------------------------------------------------
from colab_utils import (auth_wandb, auth_hf, setup_hf_cache, prefetch_models,
                          fetch_training_data, run_pipeline)

# -- HF model cache ------------------------------------------------------------
if KAGGLE_HF_DATASET and os.path.isdir(KAGGLE_HF_DATASET):
    os.environ["HF_HOME"]            = KAGGLE_HF_DATASET
    os.environ["TRANSFORMERS_CACHE"] = KAGGLE_HF_DATASET
    for _flag in ("TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_OFFLINE"):
        os.environ.pop(_flag, None)
    print(f"[cache] Using attached dataset: {KAGGLE_HF_DATASET}")
else:
    setup_hf_cache(STORAGE_ROOT)
    prefetch_models(CONFIG, GBV_DIR, STORAGE_ROOT)

# -- Auth + training data ------------------------------------------------------
auth_wandb()
auth_hf()
fetch_training_data(GBV_DIR)   # re-downloads gsm8k_train.jsonl if wiped on restart

# -- Resume pipeline (background=True so Cell 2 monitor can run in parallel) --
print()
print(f"Resuming config={CONFIG} — completed steps are skipped automatically.")
print()
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR, background=True)

In [ ]:
# =============================================================================
# Cell 2 — MONITOR  (safe to run any time, including while pipeline runs)
# Set AUTO_REFRESH = True for a live tail; interrupt the cell to stop.
# CONFIG must match the config used in Cell 0 or Cell 1.
# =============================================================================
import sys

REPO_DIR     = "/kaggle/working/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"
CONFIG       = "kaggle"  # must match Cell 0/1
AUTO_REFRESH = False     # True = live tail loop (interrupt cell to stop)
REFRESH_SECS = 20

sys.path.insert(0, f"{GBV_DIR}/deploy")
from colab_utils import monitor

monitor(STORAGE_ROOT, CONFIG, auto_refresh=AUTO_REFRESH, refresh_secs=REFRESH_SECS)

In [ ]:
# =============================================================================
# Cell 3 — SAVE  (verify artifacts; instructions to persist across sessions)
#
# Kaggle saves /kaggle/working/ when the session ends, but it is wiped on the
# NEXT session start.  To persist checkpoints permanently:
#   1. Notebook -> Data -> Output -> + New Dataset
#   2. Name it "specdist-checkpoints"
#   3. Next session: Add Data -> Your Datasets -> attach it
#      Cell 1 (Resume) restores checkpoints automatically.
# =============================================================================
import os, json, pathlib

STORAGE_ROOT = "/kaggle/working/specdist"

ckpt_dir = os.path.join(STORAGE_ROOT, "checkpoints")
db_path  = os.path.join(STORAGE_ROOT, "results.db")
log_path = os.path.join(STORAGE_ROOT, "logs", "pipeline_output.log")

status = {
    "checkpoints": os.listdir(ckpt_dir) if os.path.isdir(ckpt_dir) else [],
    "results_db_bytes": os.path.getsize(db_path) if os.path.exists(db_path) else 0,
    "log_lines": sum(1 for _ in open(log_path)) if os.path.exists(log_path) else 0,
}
with open(os.path.join(STORAGE_ROOT, "run_status.json"), "w") as f:
    json.dump(status, f, indent=2)

print(f"Artifacts at: {STORAGE_ROOT}")
print(f"  results.db   : {status['results_db_bytes']:,} bytes")
print(f"  checkpoints/ : {status['checkpoints']}")
print(f"  log lines    : {status['log_lines']}")
print()

try:
    total = sum(
        f.stat().st_size
        for f in pathlib.Path("/kaggle/working").rglob("*")
        if f.is_file()
    )
    print(f"Total /kaggle/working/ usage: {total / 1024**3:.2f} GB (limit 20 GB)")
except Exception as e:
    print(f"Could not compute disk usage: {e}")

print()
print("To persist checkpoints across sessions:")
print("  Notebook -> Data -> Output -> + New Dataset")
print("  Name it 'specdist-checkpoints' (or anything — Cell 1 tries common variants).")
print("  Attach in the next session -> Cell 1 restores everything automatically.")